# 🛠 Module 3 — Class 3: Feature Engineering (Olist Edition)

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/3](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/3)

---

## 📖 Today's story

After Classes 1-2 you have an Olist DataFrame with `total_price`, `total_freight`, `payment_type`. The boss looks at it and says:

> *"OK but this won't predict if an order is late. Where's the **distance** the package travels? Where's the **day of the week**? Where's the actual **lateness target**?"*

She's right. The columns we have are NECESSARY but not SUFFICIENT.

**Today we BUILD the columns the model actually needs.**

---

## 💡 What is feature engineering?

**Feature engineering** = creating new useful columns from the columns you already have.

It's the single biggest reason one team's model beats another's. Same data, same algorithm, but team A engineered better features → team A wins.

### 🍳 Cooking analogy

Raw ingredients (your columns):
- 🥚 eggs
- 🥛 milk
- 🌾 flour

Just dumping them in a bowl = bad omelette.

Feature engineering = **whisking eggs**, **measuring flour**, **making batter**. Same ingredients, totally different result.

---

## 🎯 Today's plan

1. **Build the target** — `is_late` (the column we want to predict)
2. **Date features** — extract hour, day-of-week, month, weekend flag
3. **The killer feature: Haversine `distance_km`** between customer ↔ seller
4. **Sanity check** — does distance correlate with `is_late`?
5. **Save** for Class 4

---

## 📦 What we'll import

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

df = pd.read_parquet('orders_step2.parquet')
print('Loaded:', df.shape)
print('\nColumns:', list(df.columns))

---

## Step 1 — Build the headline target: `is_late`

### 🎯 The definition

An order is **late** if `delivered_customer_date > estimated_delivery_date`.

It's binary: 1 (late) or 0 (on time).

### 🚨 First, drop unmodellable rows

If `delivered_customer_date` is NaN, we cannot decide late/on-time. Those rows must go to a side file (already done in Class 1). Defensive recheck:

In [ ]:
# Defensive — if any NaT slipped in, drop here
before = len(df)
df = df.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date'])
after = len(df)
print(f'Dropped {before - after} rows with NaT delivery/estimate dates.')
print(f'Remaining: {after:,} rows')

In [ ]:
# Build the target
df['is_late'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype(int)

print(f'is_late rate: {df["is_late"].mean():.3f}   ← about {df["is_late"].mean()*100:.1f}%')

print()
print('━' * 50)
print('💡 Only 7-8% of orders are late. That is IMBALANCED.')
print('   In M4-4 we will learn why accuracy lies on imbalanced data.')
print('━' * 50)

---

## Step 2 — Date features

Dates carry tons of signal. Hour-of-day, day-of-week, month — all predict shipping behaviour.

### 💡 Why decompose dates?

*"December 25, 2017, 14:30"* by itself means nothing to the model. But:
- **hour=14** → afternoon shipping
- **dayofweek=0 (Monday)** → courier just opened, fast pickup
- **month=12** → Christmas rush, slow shipping
- **is_weekend=False** → couriers operating

Each piece is a separate signal. Decompose to capture them all.

In [ ]:
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['estimate_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['days_buffer']   = df['estimate_days'] - df['delivery_days']  # negative = late

df['purchase_hour']      = df['order_purchase_timestamp'].dt.hour
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_month']     = df['order_purchase_timestamp'].dt.month
df['is_weekend']         = (df['purchase_dayofweek'] >= 5).astype(int)

df[['delivery_days','estimate_days','days_buffer',
    'purchase_hour','purchase_dayofweek','is_weekend','is_late']].head(10)

---

## Step 3 — The killer feature: Haversine distance 🌍

### Why this is the BEST feature on Olist

Brazil is huge. A customer in **Manaus** (Amazon) ordering from a seller in **São Paulo** is ~3700 km. 

The further the package, the more chances for it to be late.

**Distance is THE strongest single signal in any shipping/delivery dataset.**

### What is Haversine?

**Haversine formula** = the math for distance between two points on a sphere (the Earth).

We can't use **Euclidean** (straight-line) distance — Earth is curved. Haversine respects the curvature.

### 🍎 Kid analogy

Think of an ant on a basketball. It walks from one side to the other.
- ❌ Euclidean: "the straight line through the inside of the ball" (impossible — the ant can't tunnel)
- ✅ Haversine: "the curved path along the surface" (this is real)

For city-to-city distances on Earth, always use Haversine.

In [ ]:
# Aggregate geolocation by zip prefix (faster than per-row joins)
geo = pd.read_csv('olist_data/olist_geolocation_dataset.csv')
geo_lookup = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lon=('geolocation_lng', 'mean'),
).reset_index()
print('geo_lookup:', geo_lookup.shape)
geo_lookup.head()

In [ ]:
# Merge customer + seller lat/lon
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')
items     = pd.read_csv('olist_data/olist_order_items_dataset.csv')
sellers   = pd.read_csv('olist_data/olist_sellers_dataset.csv')

# Customer lat/lon (via customer_zip_code_prefix)
df = df.merge(customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id', how='left')
df = df.merge(geo_lookup.rename(columns={'lat':'cust_lat', 'lon':'cust_lon'}),
              left_on='customer_zip_code_prefix',
              right_on='geolocation_zip_code_prefix', how='left')
df = df.drop(columns=['geolocation_zip_code_prefix'])

# Seller lat/lon (one seller per order — pick the first item's seller)
first_seller = items.sort_values(['order_id','order_item_id']) \
                    .drop_duplicates('order_id')[['order_id','seller_id']]
df = df.merge(first_seller, on='order_id', how='left')
df = df.merge(sellers[['seller_id','seller_zip_code_prefix']], on='seller_id', how='left')
df = df.merge(geo_lookup.rename(columns={'lat':'sell_lat','lon':'sell_lon',
                                         'geolocation_zip_code_prefix':'sz'}),
              left_on='seller_zip_code_prefix', right_on='sz', how='left')
df = df.drop(columns=['sz'])

print('After merges:', df.shape)

In [ ]:
# The Haversine formula — vectorized for 100k rows in ONE numpy operation
R = 6371   # Earth radius in km

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(df['cust_lat'].values, df['cust_lon'].values,
                              df['sell_lat'].values, df['sell_lon'].values)

print('distance_km computed for all rows!')
print(df['distance_km'].describe().round(1))

---

## Step 4 — Sanity check: does distance correlate with `is_late`?

If our feature engineering is good, distance should predict lateness. Let's bin distances and look at the late rate per bin.

In [ ]:
# Bin distance into 5 ranges, look at late rate per bin
df['distance_bin'] = pd.cut(df['distance_km'], 
                             bins=[0, 100, 500, 1000, 2000, 10000],
                             labels=['<100km','100-500','500-1000','1000-2000','>2000'])

by_distance = df.groupby('distance_bin', observed=True).agg(
    n=('is_late', 'count'),
    late_rate=('is_late', 'mean'),
).round(3)

print('🌍 Late rate by distance bin:\n')
print(by_distance)

# Plot it
fig, ax = plt.subplots(figsize=(8, 4))
by_distance['late_rate'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Late rate by distance bucket — Olist orders')
ax.set_ylabel('Fraction of orders late')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

### 💡 Expected pattern

You should see a clear **monotonic increase**: closer orders are on time more often, far orders are late more often.

**That's a real signal.** Distance will be a top feature in your M4 random forest.

If you DON'T see this pattern → re-check your geolocation merge.

---

## Step 5 — Save Stage-3 output 💾

In [ ]:
# Save (drop intermediate columns we don't need downstream)
drop_cols = ['cust_lat','cust_lon','sell_lat','sell_lon',
             'customer_zip_code_prefix','seller_zip_code_prefix','distance_bin']
df_save = df.drop(columns=[c for c in drop_cols if c in df.columns])

df_save.to_parquet('orders_step3.parquet', index=False)
print(f'✅ Saved orders_step3.parquet  shape: {df_save.shape}')
print(f'\nNew columns added today:')
for c in ['is_late','delivery_days','estimate_days','days_buffer',
          'purchase_hour','purchase_dayofweek','purchase_month','is_weekend','distance_km']:
    print(f'   ✓ {c}')

---

## ✅ Quick Check

1. Why filter out rows where `order_delivered_customer_date` is NaN before computing `is_late`?
2. Why is Haversine the right distance, not Euclidean? (Use the ant-on-basketball analogy.)
3. Name two features you could engineer from `seller_id` alone (joining order history).

## 🗺️ Where these features land

| Module | Uses which features? |
| --- | --- |
| **M3-4** (next class) | All — for filter / correlation / RF importance |
| **M4-1** Linear Regression | `distance_km` predicts freight |
| **M4-2** Logistic Regression | All numeric features predict `is_late` |
| **M4-3** Random Forest | Same — RF will rank `distance_km` and `estimate_days` highest |
| **M5** RFM Segmentation | Aggregates by `customer_unique_id` using these features |
| **M6** Neural network | Same features, normalized |
| **M7** Sentiment-enriched model | Adds `nlp_sentiment` to this feature set |
| **M8** Deployed API | The /predict endpoint receives these EXACT columns from the user |

Today's features = the API contract for the rest of the program.

## 📤 Submit

### Bronze
Run all cells. Submit `orders_step3.parquet`. 
Add a markdown cell: which feature do you think will most predict `is_late`? Justify in 2 sentences.

### Gold
Engineer **at least 2 additional features** beyond what's in this notebook. Examples:
- `freight_per_item = total_freight / num_items`
- `seller_avg_delivery_days` (avg historical delivery time per seller)
- `is_high_volume_state` (state with > 5000 orders)

Justify each in 2 sentences. Save as `orders_step3_gold.parquet`.

🛠 *Great work — you turned raw columns into predictive features!*